**IMPORTING LIBRARIES**

In [ ]:
!pip install shap
!pip install streamlit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

# Sklearn models and utilities
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# SHAP for model explanations
import shap
shap.initjs() # Initializes JavaScript for visual explanations

# Streamlit for web interface (if needed later)
import streamlit as st

import warnings
warnings.filterwarnings('ignore')

print("All necessary libraries imported.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# 1. Setup Base Estimators with only XGBoost and RandomForest
estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(random_state=42))
]

# 2. Define the Stacking Classifier with Logistic Regression as the final estimator
stk_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42),
)

**LOADING DATASETS**

In [ ]:
uci_df = pd.read_csv('/content/drive/MyDrive/UCI_Dataset.csv')
uci_df.head()

In [ ]:
binary_df = pd.read_csv('/content/drive/MyDrive/Diabetes_Binary_Dataset.csv')
binary_df.head()

**INTIAL DATA INSPECTION FOR UCI**

In [ ]:
# Check for missing values and data types
print("--- Data Information ---")
print(uci_df.info())

# Specifically check for Null/NaN values
print("\n--- Missing Values Count ---")
print(uci_df.isnull().sum())
uci_df.shape
# Statistical summary of numerical columns (like Age)
print("\n--- Statistical Summary ---")
print(uci_df.describe())

In [ ]:
print(uci_df['class'].value_counts())

In [ ]:
# Check for duplicate rows
duplicates = uci_df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")
uci_df.drop_duplicates(inplace=True)

In [ ]:
# Loop to check every symptom automatically
for col in uci_df.columns:
    if col != 'Age': # Skip Age because it's a number, not Yes/No
        print(f"Distribution for {col}:")
        print(uci_df[col].value_counts())
        print("-" * 20)

**INTIAL DATA INSPECTION FOR BINARY**

In [ ]:
# Check for missing values and data types
print("--- Data Information ---")
print(binary_df.info())

# Specifically check for Null/NaN values
print("\n--- Missing Values Count ---")
print(binary_df.isnull().sum())
binary_df.shape
print(binary_df['Diabetes_binary'].value_counts())

In [ ]:
# Check for duplicate rows
duplicates = binary_df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")
# If duplicates exist, you should remove them:
binary_df = binary_df.drop_duplicates()

In [ ]:
# Loop to check distribution for all columns in binary_df
print("--- Value Counts for binary_df columns ---")
for col in binary_df.columns:
    print(f"Distribution for {col}:")
    if binary_df[col].nunique() > 20:
        display(binary_df[col].value_counts().head(10))
        print(f"(Showing top 10 values for {col})")
    else:
        display(binary_df[col].value_counts())
    print("-" * 30)

**LABEL ENCODING FOR UCI**

In [ ]:
# Initialize LabelEncoder
le = LabelEncoder()

# Loop through each column in uci_df
for column in uci_df.columns:
    # Check if the column's data type is 'object' (categorical string)
    if uci_df[column].dtype == 'object':
        # Apply label encoding to the column
        uci_df[column] = le.fit_transform(uci_df[column])

print("Label encoding applied to all categorical columns in uci_df.")
# Display the first few rows of the encoded DataFrame to verify
display(uci_df.head())
# Display data types to confirm conversion
print("\n--- Data Information After Encoding ---")
print(uci_df.info())

## Define Features and Target for UCI



In [ ]:
X_uci = uci_df.drop('class', axis=1)
y_uci = uci_df['class']

print("Features (X_uci) and target (y_uci) have been separated.")
print(f"Shape of X_uci: {X_uci.shape}")
print(f"Shape of y_uci: {y_uci.shape}")


## Define Features and Target for Binary Dataset

In [ ]:
X_binary = binary_df.drop('Diabetes_binary', axis=1)
y_binary = binary_df['Diabetes_binary']

print("Features (X_binary) and target (y_binary) have been separated.")
print(f"Shape of X_binary: {X_binary.shape}")
print(f"Shape of y_binary: {y_binary.shape}")

## Split Data into Training and Testing Sets for UCI


In [ ]:
X_train_uci, X_test_uci, y_train_uci, y_test_uci = train_test_split(X_uci, y_uci, test_size=0.2, random_state=42, stratify=y_uci)

print("UCI data split into training and testing sets:")
print(f"X_train_uci shape: {X_train_uci.shape}")
print(f"X_test_uci shape: {X_test_uci.shape}")
print(f"y_train_uci shape: {y_train_uci.shape}")
print(f"y_test_uci shape: {y_test_uci.shape}")

## Split Data into Training and Testing Sets for Binary Dataset



In [ ]:
X_train_binary, X_test_binary, y_train_binary, y_test_binary = train_test_split(X_binary, y_binary, test_size=0.2, random_state=42, stratify=y_binary)

print("Binary data split into training and testing sets:")
print(f"X_train_binary shape: {X_train_binary.shape}")
print(f"X_test_binary shape: {X_test_binary.shape}")
print(f"y_train_binary shape: {y_train_binary.shape}")
print(f"y_test_binary shape: {y_test_binary.shape}")

## Exploratory Data Analysis (EDA) for UCI Training Data

In [ ]:
print("--- Statistical Summary of X_train_uci ---")
display(X_train_uci.describe())

print("--- Value Counts for y_train_uci ---")
display(y_train_uci.value_counts())

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(X_train_uci['Age'], kde=True, bins=15, color='skyblue')
plt.title('Distribution of Age in UCI Training Data')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

print("Histogram for Age column displayed.")

In [ ]:
categorical_cols = X_train_uci.drop(columns=['Age']).columns

# Determine the number of rows and columns for the subplot grid
n_cols = 3 # Number of columns in the subplot grid
n_rows = (len(categorical_cols) + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 5, n_rows * 4))
plt.suptitle('Distribution of Categorical Features in UCI Training Data', y=1.02, fontsize=16)

for i, col in enumerate(categorical_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.countplot(x=X_train_uci[col], palette='viridis')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap

plt.show()

print("Count plots for categorical features displayed.")

## Feature Engineering for UCI Training Data



In [ ]:
bins = [0, 30, 45, 60, 75, 999]
labels = ['<30', '30-44', '45-59', '60-74', '75+']
X_train_uci['Age_Group'] = pd.cut(X_train_uci['Age'], bins=bins, labels=labels, right=False)

print("Age groups created in X_train_uci:")
display(X_train_uci['Age_Group'].value_counts().sort_index())


In [ ]:
X_test_uci['Age_Group'] = pd.cut(X_test_uci['Age'], bins=bins, labels=labels, right=False)

print("Age groups created in X_test_uci:")
display(X_test_uci['Age_Group'].value_counts().sort_index())

## EDA for Binary Training Data




In [ ]:
numerical_cols_binary = ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']

n_cols = 3 # Number of columns for the subplot grid
n_rows = (len(numerical_cols_binary) + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 6, n_rows * 5))
plt.suptitle('Distribution of Numerical Features in Binary Training Data', y=1.02, fontsize=16)

for i, col in enumerate(numerical_cols_binary):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.histplot(X_train_binary[col], kde=True, bins=20, color='teal')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap
plt.show()

print("Histograms for numerical features in X_train_binary displayed.")

## EDA for Binary Training Data




In [ ]:
print("--- Statistical Summary of X_train_binary ---")
display(X_train_binary.describe())

print("--- Value Counts for y_train_binary ---")
display(y_train_binary.value_counts())

In [ ]:
numerical_cols_binary = ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
categorical_cols_binary = [col for col in X_train_binary.columns if col not in numerical_cols_binary]

n_cols = 3 # Number of columns for the subplot grid
n_rows = (len(categorical_cols_binary) + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 5, n_rows * 4))
plt.suptitle('Distribution of Categorical Features in Binary Training Data', y=1.02, fontsize=16)

for i, col in enumerate(categorical_cols_binary):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.countplot(x=X_train_binary[col], palette='mako')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap

plt.show()

print("Count plots for categorical features in X_train_binary displayed.")

## Feature Engineering for Binary Training Data


### Create BMI Groups for Binary Data


In [ ]:
bins_bmi = [0, 18.5, 24.9, 29.9, 34.9, 39.9, 999]
labels_bmi = ['Underweight', 'Normal weight', 'Overweight', 'Obesity Class I', 'Obesity Class II', 'Obesity Class III']
X_train_binary['BMI_Group'] = pd.cut(X_train_binary['BMI'], bins=bins_bmi, labels=labels_bmi, right=True)

print("BMI groups created in X_train_binary:")
display(X_train_binary['BMI_Group'].value_counts().sort_index())

In [ ]:
X_test_binary['BMI_Group'] = pd.cut(X_test_binary['BMI'], bins=bins_bmi, labels=labels_bmi, right=True)

print("BMI groups created in X_test_binary:")
display(X_test_binary['BMI_Group'].value_counts().sort_index())


## EDA for UCI Training Data



In [ ]:
plt.figure(figsize=(8, 8))
y_train_uci.value_counts().plot.pie(autopct='%1.1f%%', startangle=90, colors=['lightcoral', 'lightskyblue'])
plt.title('Distribution of Target Variable (y_train_uci)')
plt.ylabel('') # Hide the default 'class' label on the y-axis
plt.show()

print("Pie chart for y_train_uci distribution displayed.")

## EDA for UCI Training Data




In [ ]:
uci_train_combined = pd.concat([X_train_uci.drop('Age_Group', axis=1), y_train_uci], axis=1)

plt.figure(figsize=(14, 12))
sns.heatmap(uci_train_combined.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation Matrix of UCI Training Data', fontsize=16)
plt.show()

print("Correlation matrix heatmap for UCI training data displayed.")

## EDA for UCI Training Data



In [ ]:
uci_train_combined_for_plots = pd.concat([X_train_uci, y_train_uci], axis=1)

# Identify categorical columns for plotting (all except original 'Age')
categorical_features_uci = [col for col in X_train_uci.columns if col != 'Age']

# Determine the number of rows and columns for the subplot grid
n_cols = 3 # Number of columns in the subplot grid
n_rows = (len(categorical_features_uci) + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 6, n_rows * 4))
plt.suptitle('Categorical Feature Distribution vs. Target in UCI Training Data', y=1.02, fontsize=16)

for i, col in enumerate(categorical_features_uci):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.countplot(data=uci_train_combined_for_plots, x=col, hue='class', palette='viridis')
    plt.title(f'{col} vs. Class')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend(title='Class', labels=['Negative', 'Positive'])
    plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap

plt.show()

print("Grouped bar charts for categorical features against the target variable displayed.")

## EDA for Binary Training Data




In [ ]:
binary_train_combined = pd.concat([X_train_binary.drop('BMI_Group', axis=1), y_train_binary], axis=1)

plt.figure(figsize=(18, 15))
sns.heatmap(binary_train_combined.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation Matrix of Binary Training Data', fontsize=16)
plt.show()

print("Correlation matrix heatmap for binary training data displayed.")

## Visualize the distribution of the target variable `y_train_binary` using a donut chart.




In [ ]:
plt.figure(figsize=(8, 8))
counts = y_train_binary.value_counts()

# Create a pie chart with a white circle in the middle to make it a donut chart
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90,
        colors=['lightcoral', 'lightskyblue'], pctdistance=0.85,
        wedgeprops=dict(width=0.3, edgecolor='w'))

# Draw a white circle in the center
centre_circle = plt.Circle((0,0),0.70,fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title('Distribution of Target Variable (y_train_binary)')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

print("Donut chart for y_train_binary distribution displayed.")

## Visualize BMI distribution using a box plot for Binary Training Data




In [ ]:
binary_train_combined_for_bmi = pd.concat([X_train_binary.drop(columns=['BMI_Group'], errors='ignore'), y_train_binary], axis=1)

plt.figure(figsize=(8, 6))
sns.boxplot(data=binary_train_combined_for_bmi, x='Diabetes_binary', y='BMI', palette='coolwarm')
plt.title('BMI Distribution by Diabetes Status in Binary Training Data')
plt.xlabel('Diabetes Status (0: No, 1: Yes)')
plt.ylabel('BMI')
plt.xticks([0, 1], ['No Diabetes', 'Diabetes'])
plt.grid(axis='y', alpha=0.75)
plt.show()

print("Box plot of BMI distribution by Diabetes Status displayed.")

## Visualize PhysActivity distribution using a violin plot for Binary Training Data



In [ ]:
binary_train_combined_for_phys_activity = pd.concat([X_train_binary.drop(columns=['BMI_Group'], errors='ignore'), y_train_binary], axis=1)

plt.figure(figsize=(8, 6))
sns.violinplot(data=binary_train_combined_for_phys_activity, x='Diabetes_binary', y='PhysActivity', palette='plasma')
plt.title('PhysActivity Distribution by Diabetes Status in Binary Training Data')
plt.xlabel('Diabetes Status (0: No, 1: Yes)')
plt.ylabel('Physical Activity')
plt.xticks([0, 1], ['No Diabetes', 'Diabetes'])
plt.grid(axis='y', alpha=0.75)
plt.show()

print("Violin plot of PhysActivity distribution by Diabetes Status displayed.")

## EDA for Binary Training Data


In [ ]:
binary_train_combined_for_plots = pd.concat([X_train_binary.drop(columns=['BMI_Group'], errors='ignore'), y_train_binary], axis=1)

# Reuse the previously defined list of categorical columns for binary data
numerical_cols_binary = ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
categorical_cols_binary = [col for col in X_train_binary.columns if col not in numerical_cols_binary and col != 'BMI_Group']

# Determine the number of rows and columns for the subplot grid
n_cols = 3 # Number of columns in the subplot grid
n_rows = (len(categorical_cols_binary) + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 6, n_rows * 5))
plt.suptitle('Categorical Feature Distribution vs. Target in Binary Training Data', y=1.02, fontsize=16)

for i, col in enumerate(categorical_cols_binary):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.countplot(data=binary_train_combined_for_plots, x=col, hue='Diabetes_binary', palette='mako')
    plt.title(f'Distribution of {col} by Diabetes Status')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend(title='Diabetes Status', labels=['No Diabetes', 'Diabetes'])
    plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap

plt.show()

print("Grouped bar charts for categorical features against the target variable displayed.")

### Model Selection for UCI Dataset

In [ ]:
print("--- Model Selection Conclusion for UCI Dataset ---")
print("Optimal Parameters: final_estimator__C: 1.0, rf__n_estimators: 100, xgb__n_estimators: 50")
print(f"Test Set Metrics (after tuning):\n  Accuracy: 0.9216\n  Precision: 1.0000\n  Recall: 0.8857\n  F1-score: 0.9394")
print("\nConclusion: The tuned stacking model is a highly suitable choice for the UCI dataset due to its excellent performance, particularly its perfect precision and strong recall, making it reliable for diabetes prediction in this context.")

### Model Selection for Binary Dataset

In [ ]:
print("--- Model Selection Conclusion for Binary Dataset ---")
print("Optimal Parameters: final_estimator__C: 0.1, rf__n_estimators: 100, xgb__n_estimators: 50")
print(f"Test Set Metrics (after tuning):\n  Accuracy: 0.8543\n  Precision: 0.5647\n  Recall: 0.2059\n  F1-score: 0.3017")
print("\nConclusion: The tuned stacking model is NOT a suitable choice for the Binary dataset in its current form due to very low recall and F1-score. This indicates a significant inability to correctly identify positive diabetes cases, likely stemming from severe class imbalance.")
print("\nNext Steps: Focus on addressing severe class imbalance through resampling, cost-sensitive learning, or exploring alternative models robust to imbalance to enhance its ability to correctly identify positive diabetes cases.")

## Addressing Class Imbalance in Binary Dataset with SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print(f"Original class distribution: {Counter(y_train_binary)}")

# Drop the 'BMI_Group' column from X_train_binary before applying SMOTE
# as SMOTE expects numerical input and this column contains string values.
X_train_binary_smote = X_train_binary.drop(columns=['BMI_Group'], errors='ignore')

smote = SMOTE(random_state=42)
X_train_binary_resampled, y_train_binary_resampled = smote.fit_resample(X_train_binary_smote, y_train_binary)

print(f"Resampled class distribution: {Counter(y_train_binary_resampled)}")
print("SMOTE applied to X_train_binary and y_train_binary.")

## Retrain and Evaluate Model on SMOTE-resampled Binary Data

In [ ]:
# Re-initialize stk_model for consistency with the model selection process
# Ensure it's a StackingClassifier as per the original model selection cells
estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(random_state=42))
]
stk_model_resampled_binary = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42),
)

# Fit the model on the resampled training data
stk_model_resampled_binary.fit(X_train_binary_resampled, y_train_binary_resampled)

print("Stacking model trained on SMOTE-resampled binary training data.")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Drop 'BMI_Group' from X_test_binary before prediction to match training features
X_test_binary_processed = X_test_binary.drop(columns=['BMI_Group'], errors='ignore')

# Make predictions on the original (unresampled) test set with processed features
y_pred_binary_resampled = stk_model_resampled_binary.predict(X_test_binary_processed)
y_proba_binary_resampled = stk_model_resampled_binary.predict_proba(X_test_binary_processed)

# Calculate evaluation metrics
accuracy_resampled = accuracy_score(y_test_binary, y_pred_binary_resampled)
precision_resampled = precision_score(y_test_binary, y_pred_binary_resampled)
recall_resampled = recall_score(y_test_binary, y_pred_binary_resampled)
f1_resampled = f1_score(y_test_binary, y_pred_binary_resampled)

print(f"--- Metrics After SMOTE Resampling ---")
print(f"Accuracy: {accuracy_resampled:.4f}")
print(f"Precision: {precision_resampled:.4f}")
print(f"Recall: {recall_resampled:.4f}")
print(f"F1-score: {f1_resampled:.4f}")

# Plot Confusion Matrix
cm_resampled = confusion_matrix(y_test_binary, y_pred_binary_resampled)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_resampled, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix for Binary Data (After SMOTE)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Plot ROC-AUC Curve
fpr_resampled, tpr_resampled, _ = roc_curve(y_test_binary, y_proba_binary_resampled[:, 1])
roc_auc_resampled = roc_auc_score(y_test_binary, y_proba_binary_resampled[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(fpr_resampled, tpr_resampled, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_resampled:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for Binary Data (After SMOTE)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print("Evaluation metrics, confusion matrix, and ROC-AUC curve displayed for Binary data after SMOTE.")

In [ ]:
X_train_uci = pd.get_dummies(X_train_uci, columns=['Age_Group'], drop_first=True)
X_test_uci = pd.get_dummies(X_test_uci, columns=['Age_Group'], drop_first=True)

stk_model.fit(X_train_uci, y_train_uci)

print("Stacking model (stk_model) successfully trained on UCI training data after one-hot encoding Age_Group.")

## Evaluate Stacking Model on UCI Data & Generate Plots



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score

# 1. Make predictions
y_pred_uci = stk_model.predict(X_test_uci)
y_proba_uci = stk_model.predict_proba(X_test_uci)

# 2. Calculate evaluation metrics
accuracy = accuracy_score(y_test_uci, y_pred_uci)
precision = precision_score(y_test_uci, y_pred_uci)
recall = recall_score(y_test_uci, y_pred_uci)
f1 = f1_score(y_test_uci, y_pred_uci)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

In [ ]:
cm = confusion_matrix(y_test_uci, y_pred_uci)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix for UCI Data')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Confusion matrix for UCI data displayed.")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_uci, y_proba_uci[:, 1])
roc_auc = roc_auc_score(y_test_uci, y_proba_uci[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for UCI Data')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print("ROC-AUC curve for UCI data displayed.")

## Evaluate Stacking Model on UCI Data & Generate Plots




In [ ]:
feature_importances = []

# Iterate through the base estimators and extract feature importances
# stk_model.named_estimators_ contains the *fitted* base estimators keyed by name
for name, estimator in stk_model.named_estimators_.items():
    if hasattr(estimator, 'feature_importances_'):
        feature_importances.append(estimator.feature_importances_)
    else:
        # For models like LogisticRegression, feature_importances_ might not exist
        # We can approximate by coefficients if it's a linear model, but for this task,
        # we will only consider models with direct feature_importances_ (e.g., tree-based)
        print(f"Warning: Estimator '{name}' does not have feature_importances_ attribute.")

if feature_importances:
    # Calculate the mean feature importance across all collected importances
    mean_feature_importances = np.mean(feature_importances, axis=0)

    # Create a Pandas Series to map feature names to their mean importances
    feature_importance_series = pd.Series(mean_feature_importances, index=X_train_uci.columns)

    # Sort features by importance in descending order
    sorted_features = feature_importance_series.sort_values(ascending=False)

    # Plotting the horizontal bar chart
    plt.figure(figsize=(10, 8))
    sns.barplot(x=sorted_features.values, y=sorted_features.index, palette='viridis')
    plt.title('Feature Importance for UCI Stacking Model')
    plt.xlabel('Mean Importance')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()
else:
    print("No feature importances found from the base estimators.")

print("Feature importance horizontal bar chart for UCI data displayed.")

## Train Stacking Model on Binary Data


## Evaluate Stacking Model on Binary Data & Generate Plots



In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier

# 1. Clean & Align
# Conditionally apply get_dummies only if 'BMI_Group' column still exists
if 'BMI_Group' in X_train_binary.columns:
    X_train_binary = pd.get_dummies(X_train_binary, columns=['BMI_Group'], drop_first=True)
if 'BMI_Group' in X_test_binary.columns:
    X_test_binary = pd.get_dummies(X_test_binary, columns=['BMI_Group'], drop_first=True)

# Align columns - crucial after one-hot encoding to ensure both train and test sets
# have the same columns in the same order.
train_cols = X_train_binary.columns
X_test_binary = X_test_binary.reindex(columns=train_cols, fill_value=0)

y_train_series = y_train_binary.values.ravel() if hasattr(y_train_binary, 'values') else y_train_binary

# 2. THE SPEED FIX: Voting instead of Stacking
# This trains each model ONCE. No loops. No waiting.
stk_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42)), # Added random_state
        ('hgb', HistGradientBoostingClassifier(random_state=42)) # Added random_state
    ],
    voting='soft', # Averages the probabilities
    n_jobs=-1
)

# 3. Fit
stk_model.fit(X_train_binary, y_train_series)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score

# 1. Make predictions
y_pred_binary = stk_model.predict(X_test_binary)
y_proba_binary = stk_model.predict_proba(X_test_binary)

# 2. Calculate evaluation metrics
accuracy_binary = accuracy_score(y_test_binary, y_pred_binary)
precision_binary = precision_score(y_test_binary, y_pred_binary)
recall_binary = recall_score(y_test_binary, y_pred_binary)
f1_binary = f1_score(y_test_binary, y_pred_binary)

print(f"Accuracy for Binary Data: {accuracy_binary:.4f}")
print(f"Precision for Binary Data: {precision_binary:.4f}")
print(f"Recall for Binary Data: {recall_binary:.4f}")
print(f"F1-score for Binary Data: {f1_binary:.4f}")

In [ ]:
cm_binary = confusion_matrix(y_test_binary, y_pred_binary)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_binary, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix for Binary Data')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Confusion matrix for Binary data displayed.")

In [ ]:
fpr_binary, tpr_binary, thresholds_binary = roc_curve(y_test_binary, y_proba_binary[:, 1])
roc_auc_binary = roc_auc_score(y_test_binary, y_proba_binary[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr_binary, tpr_binary, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_binary:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for Binary Data')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print("ROC-AUC curve for Binary data displayed.")

In [ ]:
feature_importances_binary = []

# Iterate through the base estimators and extract feature importances
# stk_model.named_estimators_ contains the *fitted* base estimators keyed by name
for name, estimator in stk_model.named_estimators_.items():
    if hasattr(estimator, 'feature_importances_'):
        feature_importances_binary.append(estimator.feature_importances_)
    else:
        # For models like LogisticRegression, feature_importances_ might not exist
        print(f"Warning: Estimator '{name}' does not have feature_importances_ attribute.")

if feature_importances_binary:
    # Calculate the mean feature importance across all collected importances
    mean_feature_importances_binary = np.mean(feature_importances_binary, axis=0)

    # Create a Pandas Series to map feature names to their mean importances
    feature_importance_series_binary = pd.Series(mean_feature_importances_binary, index=X_train_binary.columns)

    # Sort features by importance in descending order
    sorted_features_binary = feature_importance_series_binary.sort_values(ascending=False)

    # Plotting the horizontal bar chart
    plt.figure(figsize=(10, 8))
    sns.barplot(x=sorted_features_binary.values, y=sorted_features_binary.index, palette='magma')
    plt.title('Feature Importance for Binary Stacking Model')
    plt.xlabel('Mean Importance')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()
else:
    print("No feature importances found from the base estimators for Binary data.")

print("Feature importance horizontal bar chart for Binary data displayed.")

## Hyperparameter Tuning for UCI Model



In [ ]:
from sklearn.model_selection import GridSearchCV

# Re-initialize stk_model as a StackingClassifier for UCI tuning
# This ensures we are tuning the correct model type with the correct parameters
estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(random_state=42))
]
stk_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42),
)

# Define parameter grid for the StackingClassifier
param_grid_uci = {
    'rf__n_estimators': [50, 100],
    'xgb__n_estimators': [50, 100],
    'final_estimator__C': [0.1, 1.0]
}

# Instantiate GridSearchCV
grid_search_uci = GridSearchCV(estimator=stk_model,
                               param_grid=param_grid_uci,
                               cv=3, # 3-fold cross-validation
                               scoring='accuracy',
                               n_jobs=-1, # Use all available CPU cores
                               verbose=2)

# Fit GridSearchCV to the UCI training data
grid_search_uci.fit(X_train_uci, y_train_uci)

print("GridSearchCV completed for UCI model.")

In [ ]:
print("Best parameters found for UCI model:")
display(grid_search_uci.best_params_)

print("Best cross-validation accuracy for UCI model:")
display(grid_search_uci.best_score_)

In [ ]:
tuned_stk_model_uci = grid_search_uci.best_estimator_

y_pred_uci_tuned = tuned_stk_model_uci.predict(X_test_uci)
y_proba_uci_tuned = tuned_stk_model_uci.predict_proba(X_test_uci)

accuracy_tuned_uci = accuracy_score(y_test_uci, y_pred_uci_tuned)
precision_tuned_uci = precision_score(y_test_uci, y_pred_uci_tuned)
recall_tuned_uci = recall_score(y_test_uci, y_pred_uci_tuned)
f1_tuned_uci = f1_score(y_test_uci, y_pred_uci_tuned)

print(f"Tuned UCI Model Accuracy: {accuracy_tuned_uci:.4f}")
print(f"Tuned UCI Model Precision: {precision_tuned_uci:.4f}")
print(f"Tuned UCI Model Recall: {recall_tuned_uci:.4f}")
print(f"Tuned UCI Model F1-score: {f1_tuned_uci:.4f}")

## Hyperparameter Tuning for Binary Model



In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid for the StackingClassifier
param_grid_binary = {
    'rf__n_estimators': [50, 100],
    'xgb__n_estimators': [50, 100],
    'final_estimator__C': [0.1, 1.0]
}

# Instantiate GridSearchCV
grid_search_binary = GridSearchCV(estimator=stk_model,
                               param_grid=param_grid_binary,
                               cv=3, # 3-fold cross-validation
                               scoring='accuracy',
                               n_jobs=-1, # Use all available CPU cores
                               verbose=2)

# Fit GridSearchCV to the Binary training data
grid_search_binary.fit(X_train_binary, y_train_binary)

print("GridSearchCV completed for Binary model.")

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [ ]:
print("Best parameters found for Binary model:")
display(grid_search_binary.best_params_)

print("Best cross-validation accuracy for Binary model:")
display(grid_search_binary.best_score_)

In [ ]:
tuned_stk_model_binary = grid_search_binary.best_estimator_

y_pred_binary_tuned = tuned_stk_model_binary.predict(X_test_binary)
y_proba_binary_tuned = tuned_stk_model_binary.predict_proba(X_test_binary)

accuracy_tuned_binary = accuracy_score(y_test_binary, y_pred_binary_tuned)
precision_tuned_binary = precision_score(y_test_binary, y_pred_binary_tuned)
recall_tuned_binary = recall_score(y_test_binary, y_pred_binary_tuned)
f1_tuned_binary = f1_score(y_test_binary, y_pred_binary_tuned)

print(f"Tuned Binary Model Accuracy: {accuracy_tuned_binary:.4f}")
print(f"Tuned Binary Model Precision: {precision_tuned_binary:.4f}")
print(f"Tuned Binary Model Recall: {recall_tuned_binary:.4f}")
print(f"Tuned Binary Model F1-score: {f1_tuned_binary:.4f}")

In [ ]:
explainer_uci = shap.Explainer(tuned_stk_model_uci.predict, X_test_uci, feature_names=X_test_uci.columns)
print("SHAP Explainer initialized for UCI model.")

In [ ]:
X_test_uci_float = X_test_uci.astype(float)

# It's good practice to use a sample of the training data as background for SHAP Explainer
# Also, use predict_proba for classification models for better explanations of class probabilities
explainer_uci = shap.Explainer(tuned_stk_model_uci.predict_proba, X_train_uci.sample(100, random_state=42).astype(float), feature_names=X_test_uci_float.columns)
print("SHAP Explainer initialized for UCI model.")

In [ ]:
shap_values_uci = explainer_uci(X_test_uci_float)
print("SHAP values calculated for UCI model.")

In [ ]:
shap.summary_plot(shap_values_uci.values[:, :, 1], X_test_uci_float, plot_type="bar")
plt.title('SHAP Summary Plot for UCI Model (Positive Class)')
plt.show()

print("SHAP summary plot for UCI model displayed.")

## Explain Binary Model with SHAP




In [ ]:
X_test_binary_float = X_test_binary.astype(float)

print("X_test_binary converted to float type.")

In [ ]:
explainer_binary = shap.Explainer(tuned_stk_model_binary.predict_proba, X_train_binary.sample(100, random_state=42).astype(float), feature_names=X_test_binary_float.columns)
print("SHAP Explainer initialized for Binary model.")

In [ ]:
shap_values_binary = explainer_binary(X_test_binary_float)
print("SHAP values calculated for Binary model.")